In [ ]:
# Pinned Hugging Face training stack (run once in a fresh Colab GPU session)
# Restart Runtime after package installation, then re-run the data setup.
%pip install -q "transformers==4.57.1" "huggingface-hub==0.36.2" "trl==0.23.1" "peft==0.17.1" "accelerate" "bitsandbytes>=0.45" "datasets" "evaluate" "pandas" "matplotlib" "wordcloud"

# Transformer-Based Question Answering — DistilBERT & Qwen3

Two independent SQuAD 2.0 question-answering tracks:

1. **DistilBERT**, an extractive span-prediction model. Train and evaluate on a reproducible pilot.
2. **Qwen3-0.6B**, a generative model adapted with NF4 4-bit QLoRA and completion-only supervised fine-tuning.

**Experiment honesty:** Model training, quantization and dataset downloading require a Colab GPU and network. Saved outputs in the original notebook are historical and were cleared for this clean version. Recorded Qwen3 benchmark results are documented in the README, not fabricated as new notebook outputs.

**Order:** Install pinned dependencies → restart kernel → load SQuAD → choose a model track. The Qwen3 code is opt-in. DistilBERT predicts answer spans and Qwen3 generates short grounded answers.

In [ ]:
# Part1: Setup and Load Data
import torch
import numpy as np
import random
import matplotlib.pyplot as plt
from datasets import load_dataset

# 1. set seed=42
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# 2. load SQuAD 2.0 dataset
print("Loading SQuAD 2.0 dataset...")
dataset = load_dataset("squad_v2")
print(dataset)

In [ ]:
# Part 2: Samples and Visualization
from wordcloud import WordCloud
import seaborn as sns

train_data = dataset['train']

# 1. 5 Samples
print("\n5 Sample Entries")
for i in range(5):
    sample = train_data[i]
    print(f"Index: {i}")
    print(f"Context: {sample['context'][:100]}...")
    print(f"Question: {sample['question']}")
    ans = sample['answers']['text'][0] if sample['answers']['text'] else "No Answer"
    print(f"Answer: {ans}")
    print("-" * 30)

# 2. Answerable and Unanswerable
total = len(train_data)
has_answer = sum(1 for x in train_data if len(x['answers']['text']) > 0)
no_answer = total - has_answer

print(f"\nDistribution Statistics")
print(f"Total Training Samples: {total}")
print(f"Answerable Questions: {has_answer} ({has_answer/total:.2%})")
print(f"Unanswerable Questions: {no_answer} ({no_answer/total:.2%})")

# 3. Calculate Passage Length, Visualize Histogram
context_lengths = [len(x['context'].split()) for x in train_data]
avg_length = sum(context_lengths) / len(context_lengths)
print(f"Average Passage Length (words): {avg_length:.2f}")

plt.figure(figsize=(10, 5))
plt.hist(context_lengths, bins=50, color='skyblue', edgecolor='black')
plt.title('Distribution of Passage Lengths')
plt.xlabel('Number of Words')
plt.ylabel('Frequency')
plt.axvline(x=384, color='r', linestyle='--', label='Max Length Limit (384)') # 标注出作业限制
plt.legend()
plt.show()

# 4. Word Cloud for Questions
print("\nGenerating Word Cloud for Questions...")

questions_list = train_data[:10000]['question']
text = " ".join(questions_list)

wordcloud = WordCloud(width=800, height=400, background_color='white').generate(text)

plt.figure(figsize=(10, 5))
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis('off')
plt.title('Word Cloud of Questions')
plt.show()

In [ ]:
# Part 3: Preprocessing
from transformers import AutoTokenizer

# 1. Load Fast Tokenizer
model_checkpoint = "distilbert-base-uncased"
try:
    tokenizer = AutoTokenizer.from_pretrained(model_checkpoint, use_fast=True)
except:
    tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

# 2. Set Parameters
max_length = 384          # Total length (Passage + Question + Special tokens)
max_question_length = 32  # Max question length
doc_stride = 128          # Sliding window stride

def preprocess_function(examples):

    questions = [q.strip() for q in examples["question"]]

    # Tokenize
    tokenized_examples = tokenizer(
        questions,
        examples["context"],
        truncation="only_second",
        max_length=max_length,
        stride=doc_stride,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length",
    )

    # Handle mapping key names
    if "overflow_to_sample_mapping" in tokenized_examples:
        sample_mapping = tokenized_examples.pop("overflow_to_sample_mapping")
    else:
        sample_mapping = tokenized_examples.pop("overflowing_tokens_mapping")

    offset_mapping = tokenized_examples.pop("offset_mapping")

    tokenized_examples["start_positions"] = []
    tokenized_examples["end_positions"] = []

    for i, offsets in enumerate(offset_mapping):
        input_ids = tokenized_examples["input_ids"][i]
        cls_index = input_ids.index(tokenizer.cls_token_id)
        sequence_ids = tokenized_examples.sequence_ids(i)
        sample_index = sample_mapping[i]
        answers = examples["answers"][sample_index]

        if len(answers["answer_start"]) == 0:     # If the answer_start list is empty, it indicates that the sample is an unanswerable question
            tokenized_examples["start_positions"].append(cls_index)
            tokenized_examples["end_positions"].append(cls_index)   # Both the start and end positions of the answer are set to the index of the [CLS] token
        else:
            start_char = answers["answer_start"][0]
            end_char = start_char + len(answers["text"][0])

            token_start_index = 0
            while sequence_ids[token_start_index] != 1:
                token_start_index += 1
            token_end_index = len(input_ids) - 1
            while sequence_ids[token_end_index] != 1:
                token_end_index -= 1
            # Compare the start character position of the first token with the end character position of the last token in the current span
            # If the answer range (from start_char to end_char) is not fully contained within this span
            if not (offsets[token_start_index][0] <= start_char and offsets[token_end_index][1] >= end_char):
                tokenized_examples["start_positions"].append(cls_index)
                tokenized_examples["end_positions"].append(cls_index)   # The start_positions and end_positions are pointed to the [CLS] token
            else:
                while token_start_index < len(offsets) and offsets[token_start_index][0] <= start_char:
                    token_start_index += 1
                tokenized_examples["start_positions"].append(token_start_index - 1)
                while offsets[token_end_index][1] >= end_char:
                    token_end_index -= 1
                tokenized_examples["end_positions"].append(token_end_index + 1)

    return tokenized_examples


DO_SUBSAMPLE = True
SAMPLE_SIZE = 2000

if DO_SUBSAMPLE:
    print(f"Subsampling dataset to {SAMPLE_SIZE} examples for speed...")
    # 'select' returns a new subset of the dataset
    small_train = dataset["train"].shuffle(seed=42).select(range(SAMPLE_SIZE))
    small_val = dataset["validation"].shuffle(seed=42).select(range(min(SAMPLE_SIZE, len(dataset["validation"]))))

    # Reconstruct the DatasetDict
    from datasets import DatasetDict
    dataset_to_process = DatasetDict({"train": small_train, "validation": small_val})
else:
    print("Using FULL dataset (this will take longer to train)...")
    dataset_to_process = dataset

tokenized_datasets = dataset_to_process.map(preprocess_function, batched=True, remove_columns=dataset["train"].column_names)
print("Preprocessing complete!")

# Print current counts
print(f"Train set size: {len(tokenized_datasets['train'])}")
print(f"Validation set size: {len(tokenized_datasets['validation'])}")

In [ ]:
# Part 4: Model Initialization and Training
from transformers import AutoModelForQuestionAnswering, TrainingArguments, Trainer, DefaultDataCollator
import torch
# Extractive QA using DistilBERT
# 1. Load DistilBERT
# Fix random initialization of the QA prediction head for the baseline comparison.
torch.manual_seed(42)
model_checkpoint = "distilbert-base-uncased"
print(f"Loading model: {model_checkpoint}...")
model = AutoModelForQuestionAnswering.from_pretrained(model_checkpoint)

# 2. Set Hyperparameters
batch_size = 16

args = TrainingArguments(
    output_dir="./squad_model_output",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=50,
    fp16=torch.cuda.is_available(),
    push_to_hub=False,
    optim="adamw_torch",
    seed=42,
    report_to="none",
)

# 3. Initialize Trainer
# Hugging Face Trainer API
data_collator = DefaultDataCollator()

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    processing_class=tokenizer,
)

# 4. Start Training
trainer.train()

# 5. Save Final Model
trainer.save_model("./squad_finetuned_model")
tokenizer.save_pretrained("./squad_finetuned_model")
print("Training finished!")

In [ ]:
# Part 5: Plotting Training Curves
import pandas as pd
import matplotlib.pyplot as plt

# Extract history data from trainer logs
history = trainer.state.log_history
df = pd.DataFrame(history)

# Filter out training loss and validation loss
train_loss = df[df['loss'].notnull()][['step', 'loss']]
eval_loss = df[df['eval_loss'].notnull()][['step', 'eval_loss']]

plt.figure(figsize=(12, 6))

# Training Loss
plt.plot(train_loss['step'], train_loss['loss'], label='Training Loss', alpha=0.6)

# Validation Loss
plt.plot(eval_loss['step'], eval_loss['eval_loss'], label='Validation Loss', color='black', marker='o', linewidth=2)

plt.xlabel('Training Steps')
plt.ylabel('Loss')
plt.title('Training and Validation Loss Curves')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# Part 10: SQuAD 2.0 evaluation — EM / Token F1 on identical examples
# IMPORTANT: Values are generated only by running this cell. Previous printed
# losses and manually prepared tables are NOT substitute metrics.
from pathlib import Path
import gc
import random
import pandas as pd
import torch
import evaluate
from transformers import (
    AutoModelForQuestionAnswering, AutoTokenizer
)

RUN_QA_BENCHMARK = True
EVAL_PER_CLASS = 40        # align sample IDs with the Qwen3 40+40 pilot
QA_BENCHMARK_SEED = 43
QA_LOCAL_PATH = "./squad_finetuned_model"
QA_BASE_CHECKPOINT = "distilbert-base-uncased"
QA_REFERENCE_CHECKPOINT = "distilbert-base-cased-distilled-squad"

def make_balanced_qa_eval_samples(validation, per_class=50, seed=42):
    rng = random.Random(seed)
    indices = list(range(len(validation)))
    rng.shuffle(indices)
    answerable, unanswerable = [], []
    for idx in indices:
        example = validation[idx]
        if example["answers"]["text"]:
            if len(answerable) < per_class:
                answerable.append(example)
        else:
            if len(unanswerable) < per_class:
                unanswerable.append(example)
        if len(answerable) == per_class and len(unanswerable) == per_class:
            break
    if len(answerable) < per_class or len(unanswerable) < per_class:
        raise ValueError("Not enough examples to create the balanced pilot set.")
    selected = answerable + unanswerable
    rng.shuffle(selected)
    return selected

def extract_answer_span(
    model, tokenizer, question, context, device,
    max_seq_len=384, doc_stride=128, max_answer_len=30,
    null_score_diff_threshold=0.0,
):
    """Extract answers using logits + offset mapping (Transformers v5-compatible).

    A single question/context can produce multiple overlapping windows.
    The [CLS] score represents the no-answer hypothesis. The default score
    threshold is uncalibrated and intended for pilot experiments only.
    """
    if not question.strip() or not context.strip():
        return ""
    if not tokenizer.is_fast:
        raise ValueError("The model requires a fast tokenizer with offset mapping.")

    features = tokenizer(
        question.strip(), context,
        truncation="only_second", max_length=max_seq_len,
        stride=doc_stride, return_overflowing_tokens=True,
        return_offsets_mapping=True, padding="max_length",
        return_tensors="pt",
    )
    best_answer = ""
    best_span_score = float("-inf")
    best_null_score = float("inf")
    model.eval()

    for feature_i in range(features["input_ids"].shape[0]):
        token_ids = features["input_ids"][feature_i].tolist()
        offsets = features["offset_mapping"][feature_i].tolist()
        sequence_ids = features.sequence_ids(feature_i)
        attn_mask = features["attention_mask"][feature_i].tolist()

        if tokenizer.cls_token_id not in token_ids:
            raise ValueError("Expected [CLS] token for no-answer scoring.")
        cls_i = token_ids.index(tokenizer.cls_token_id)

        inputs = {
            "input_ids": features["input_ids"][feature_i:feature_i + 1].to(device),
            "attention_mask": features["attention_mask"][feature_i:feature_i + 1].to(device),
        }
        with torch.inference_mode():
            logits = model(**inputs)
        start_logits = logits.start_logits[0].float().cpu().tolist()
        end_logits = logits.end_logits[0].float().cpu().tolist()
        best_null_score = min(
            best_null_score, start_logits[cls_i] + end_logits[cls_i]
        )

        valid_positions = [
            pos for pos, segment in enumerate(sequence_ids)
            if segment == 1 and attn_mask[pos] == 1
            and offsets[pos][0] < offsets[pos][1]
        ]
        valid_set = set(valid_positions)

        for start_i in valid_positions:
            for end_i in range(start_i, min(start_i + max_answer_len, len(end_logits))):
                if end_i not in valid_set:
                    continue
                char_start = offsets[start_i][0]
                char_end = offsets[end_i][1]
                if char_start >= char_end:
                    continue
                score = start_logits[start_i] + end_logits[end_i]
                if score > best_span_score:
                    best_span_score = score
                    best_answer = context[char_start:char_end]

    if best_span_score == float("-inf"):
        return ""
    if best_null_score - best_span_score > null_score_diff_threshold:
        return ""
    return best_answer.strip()


def evaluate_qa_model(model, tokenizer_name, examples, metric):
    """Compare QA models on an identical SQuAD v2 pilot set."""
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    created_here = isinstance(model, (str, Path))
    if created_here:
        model = AutoModelForQuestionAnswering.from_pretrained(str(model))
    tokenizer = AutoTokenizer.from_pretrained(tokenizer_name, use_fast=True)
    model = model.to(device).eval()
    predictions = []
    references = []

    for example in examples:
        predicted_text = extract_answer_span(
            model, tokenizer, example["question"], example["context"], device,
            max_seq_len=384, doc_stride=128, max_answer_len=30,
            null_score_diff_threshold=0.0,
        )
        predictions.append({
            "id": example["id"],
            "prediction_text": predicted_text,
            # Binary abstention output, not a calibrated null probability.
            "no_answer_probability": float(predicted_text == ""),
        })
        references.append({"id": example["id"], "answers": example["answers"]})

    results = metric.compute(
        predictions=predictions, references=references,
        no_answer_threshold=0.5,
    )
    if created_here:
        del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return results

qa_results = None
if RUN_QA_BENCHMARK:
    if not Path(QA_LOCAL_PATH).is_dir():
        raise FileNotFoundError(
            f"No fine-tuned QA checkpoint at {QA_LOCAL_PATH}. Run Part 4 first."
        )
    qa_examples = make_balanced_qa_eval_samples(
        dataset["validation"], per_class=EVAL_PER_CLASS, seed=QA_BENCHMARK_SEED
    )
    squad_metric = evaluate.load("squad_v2")

    # The initial QA prediction head has random weights, while the encoder
    # loads pre-trained weights. This is the non-fine-tuned starting baseline.
    # The training cell resets the seed before initializing the same model.
    torch.manual_seed(42)
    untuned_model = AutoModelForQuestionAnswering.from_pretrained(
        QA_BASE_CHECKPOINT
    )
    qa_results = {}
    qa_results["Untuned DistilBERT (random QA head)"] = evaluate_qa_model(
        untuned_model, QA_BASE_CHECKPOINT, qa_examples, squad_metric
    )
    del untuned_model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    # Same examples, processing window and metric for a meaningful comparison.
    qa_results["Fine-tuned DistilBERT (this notebook)"] = evaluate_qa_model(
        QA_LOCAL_PATH, QA_LOCAL_PATH, qa_examples, squad_metric
    )
    # Additional reference: already fine-tuned by the original publisher.
    qa_results["Published SQuAD QA model (reference)"] = evaluate_qa_model(
        QA_REFERENCE_CHECKPOINT, QA_REFERENCE_CHECKPOINT,
        qa_examples, squad_metric
    )

    qa_comparison = pd.DataFrame([
        {
            "model": label,
            "EM (%)": round(result["exact"], 2),
            "Token F1 (%)": round(result["f1"], 2),
            "No-answer F1 (%)": round(result.get("NoAns_f1", float("nan")), 2),
            "examples": result["total"],
        }
        for label, result in qa_results.items()
    ])
    display(qa_comparison)
    qa_comparison.to_csv("qa_metrics_pilot.csv", index=False)
    print("Pilot EM / F1 saved: qa_metrics_pilot.csv")
    print("Balanced 80-example subset, NOT the official full SQuAD benchmark.")
    print("For complete evaluation, process all SQuAD 2.0 validation examples")
    print("and calibrate the no-answer score threshold on held-out data.")


## Qwen3-0.6B — supervised fine-tuning with 4-bit QLoRA

Set `RUN_QWEN_QLORA = True` to train. You need a CUDA GPU and the pinned dependencies above. The prompt teaches short answers grounded in a supplied passage; unanswerable examples use an explicit refusal target. Validation samples must be kept separate from training samples. Do not equate token accuracy with exact-match QA accuracy.

In [ ]:
# Qwen3 experiment switch — no installs inside conditional training code.
RUN_QWEN_QLORA = False
if RUN_QWEN_QLORA:
    import torch
    if not torch.cuda.is_available():
        raise RuntimeError("Select a CUDA GPU runtime in Colab before QLoRA.")
    print("Qwen3 QLoRA experiment enabled.")
else:
    print("Qwen3 QLoRA is disabled; set RUN_QWEN_QLORA=True to train.")

In [ ]:
# Qwen3 QLoRA experiment (run only after dependencies are available).
# Separate generative alternative to DistilBERT.
if RUN_QWEN_QLORA:
    import time
    import torch
    import random
    from datasets import Dataset
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
    from peft import LoraConfig, prepare_model_for_kbit_training
    from trl import SFTConfig, SFTTrainer

    QWEN_ID = "Qwen/Qwen3-0.6B"
    qwen_tok = AutoTokenizer.from_pretrained(QWEN_ID)
    if qwen_tok.pad_token is None:
        qwen_tok.pad_token = qwen_tok.eos_token

    SYSTEM_RULES = (
        "Answer using only the provided passage. Give a short factual answer. "
        "If the answer is absent, say: No answer in the provided context."
    )

    def qwen_prompt_completion(example):
        # Token-based context window; center it around the gold answer
        # whenever an answer exists instead of hiding answer evidence.
        full_context = example["context"]
        answer_texts = example["answers"]["text"]
        answer_starts = example["answers"]["answer_start"]
        offsets = qwen_tok(
            full_context, add_special_tokens=False, return_offsets_mapping=True
        )["offset_mapping"]
        max_context_tokens = 300
        if len(offsets) <= max_context_tokens:
            context = full_context
        else:
            first_token = 0
            if answer_starts:
                answer_start = answer_starts[0]
                answer_token = next(
                    (i for i, span in enumerate(offsets) if span[1] > answer_start),
                    0
                )
                first_token = min(
                    max(0, answer_token - 100), len(offsets) - max_context_tokens
                )
            last_token = first_token + max_context_tokens - 1
            context = full_context[
                offsets[first_token][0]:offsets[last_token][1]
            ]
        messages = [
            {"role": "system", "content": SYSTEM_RULES},
            {"role": "user", "content": (
                f"Passage:\n{context}\n\nQuestion:\n{example['question']}"
            )},
        ]
        prompt = qwen_tok.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=False,
        )
        # For answerable rows, require the original annotated character span
        # to survive the crop. Never silently rewrite a positive label to
        # a negative one just because the context had to be shortened.
        if answer_texts and answer_starts:
            span_start = answer_starts[0]
            span_end = span_start + len(answer_texts[0])
            if len(offsets) <= max_context_tokens:
                window_start_char, window_end_char = 0, len(full_context)
            else:
                window_start_char = offsets[first_token][0]
                window_end_char = offsets[last_token][1]
            if not (
                window_start_char <= span_start
                and span_end <= window_end_char
            ):
                return None
            target = answer_texts[0]
        else:
            target = "No answer in the provided context."
        return {"prompt": prompt, "completion": target + qwen_tok.eos_token}

    # Standalone seeded selection; the Qwen pilot can be run without
    # executing the separate EM/F1 benchmark cell above.
    def qwen_balanced_samples(split, per_class, seed):
        rng = random.Random(seed)
        indices = list(range(len(split)))
        rng.shuffle(indices)
        answerable = []
        unanswerable = []
        for idx in indices:
            item = split[idx]
            if item["answers"]["text"]:
                if len(answerable) < per_class:
                    answerable.append(item)
            else:
                if len(unanswerable) < per_class:
                    unanswerable.append(item)
            if len(answerable) == per_class and len(unanswerable) == per_class:
                break
        if len(answerable) < per_class or len(unanswerable) < per_class:
            raise ValueError("Not enough examples for balanced Qwen3 pilot.")
        result = answerable + unanswerable
        rng.shuffle(result)
        return result

    train_samples = qwen_balanced_samples(
        dataset["train"], per_class=150, seed=42
    )
    val_samples = qwen_balanced_samples(
        dataset["validation"], per_class=40, seed=43
    )
    qwen_max_length = 512

    def fits_qwen_budget(example):
        prompt_ids = qwen_tok(
            example["prompt"], add_special_tokens=False
        )["input_ids"]
        completion_ids = qwen_tok(
            example["completion"], add_special_tokens=False
        )["input_ids"]
        return len(prompt_ids) + len(completion_ids) <= qwen_max_length

    train_rows = [
        row for row in map(qwen_prompt_completion, train_samples)
        if row is not None and fits_qwen_budget(row)
    ]
    val_rows = [
        row for row in map(qwen_prompt_completion, val_samples)
        if row is not None and fits_qwen_budget(row)
    ]
    if len(train_rows) < 100 or len(val_rows) < 30:
        raise RuntimeError("Not enough complete Qwen3 pilot examples.")
    qwen_train = Dataset.from_list(train_rows)
    qwen_eval = Dataset.from_list(val_rows)
    def count_target_types(rows):
        positive = sum(
            row["completion"].strip().replace(qwen_tok.eos_token, "").strip()
            != "No answer in the provided context."
            for row in rows
        )
        return f"{positive} answerable / {len(rows)-positive} unanswerable"

    print(f"Qwen3 pilot train: {len(qwen_train)} ({count_target_types(train_rows)})")
    print(f"Qwen3 pilot validation: {len(qwen_eval)} ({count_target_types(val_rows)})")

    gpu_major, gpu_minor = torch.cuda.get_device_capability(0)
    if gpu_major >= 8:
        compute_dtype = torch.bfloat16
    else:
        compute_dtype = torch.float16
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=compute_dtype,
    )
    qwen_model = AutoModelForCausalLM.from_pretrained(
        QWEN_ID,
        quantization_config=bnb_config,
        device_map={"": torch.cuda.current_device()},
        dtype=compute_dtype,
    )
    qwen_model = prepare_model_for_kbit_training(qwen_model)

    qwen_lora = LoraConfig(
        task_type="CAUSAL_LM",
        r=8,
        lora_alpha=16,
        lora_dropout=0.05,
        target_modules="all-linear",
        bias="none",
    )
    qwen_args = SFTConfig(
        output_dir="./qwen3_qlora_output",
        max_length=qwen_max_length,
        completion_only_loss=True,
        num_train_epochs=1,
        per_device_train_batch_size=1,
        per_device_eval_batch_size=1,
        gradient_accumulation_steps=8,
        learning_rate=2e-4,
        warmup_ratio=0.03,
        gradient_checkpointing=True,
        fp16=(compute_dtype == torch.float16),
        bf16=(compute_dtype == torch.bfloat16),
        eval_strategy="epoch",
        save_strategy="no",
        logging_steps=10,
        report_to="none",
        seed=42,
    )
    qwen_trainer = SFTTrainer(
        model=qwen_model,
        args=qwen_args,
        train_dataset=qwen_train,
        eval_dataset=qwen_eval,
        peft_config=qwen_lora,
        processing_class=qwen_tok,
    )
    print("Training Qwen3 with 4-bit QLoRA; pilot results only.")
    print("GPU:", torch.cuda.get_device_name(torch.cuda.current_device()))
    # Peak allocated VRAM includes existing model allocations in this process.
    # For a clean per-experiment measurement, launch in a fresh GPU runtime.
    torch.cuda.reset_peak_memory_stats()
    qwen_start = time.perf_counter()
    qwen_trainer.train()
    print(f"Elapsed training time: {time.perf_counter() - qwen_start:.1f} seconds")
    print(f"GPU peak allocated: {torch.cuda.max_memory_allocated()/1024**3:.2f} GiB")
    qwen_eval_result = qwen_trainer.evaluate()
    print("Held-out pilot eval loss:", qwen_eval_result.get("eval_loss"))
    qwen_trainer.save_model("./qwen3_qlora_adapter")
    qwen_tok.save_pretrained("./qwen3_qlora_adapter")
    print("QLoRA adapter saved to ./qwen3_qlora_adapter")


In [ ]:
# Qwen3 inference demonstration (after QLoRA training).
if RUN_QWEN_QLORA:
    # qwen_trainer.model already contains the trained LoRA adapter.
    tuned_qwen = qwen_trainer.model.eval()

    def generate_grounded_qwen_answer(context, question):
        messages = [
            {"role": "system", "content": SYSTEM_RULES},
            {"role": "user", "content": f"Passage:\n{context}\n\nQuestion:\n{question}"},
        ]
        prompt = qwen_tok.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=False,
        )
        batch = qwen_tok(
            prompt, return_tensors="pt", truncation=True, max_length=450
        ).to(tuned_qwen.device)
        with torch.inference_mode():
            with torch.autocast(device_type="cuda", dtype=torch.float16):
                generated = tuned_qwen.generate(
                    **batch,
                    max_new_tokens=64,
                    do_sample=False,
                    pad_token_id=qwen_tok.eos_token_id,
                )
        new_ids = generated[0][batch["input_ids"].shape[-1]:]
        return qwen_tok.decode(new_ids, skip_special_tokens=True).strip()

    sample = val_samples[0]
    print("Question:", sample["question"])
    print("Gold answer:", sample["answers"]["text"])
    print("Qwen3 pilot output:", generate_grounded_qwen_answer(
        sample["context"], sample["question"]
    ))


## Validation checklist

- Verify the DistilBERT checkpoint is generated before running its evaluation cell.
- Confirm that the 80-example balanced DistilBERT pilot and Qwen3 pilot use the same seed and IDs before making direct comparisons.
- When running Qwen3, back up `./qwen3_qlora_adapter` outside Colab.
- The Qwen3 SFT audit and paired 80-example benchmark reported in the README were executed in a previous Colab session and **are not reproduced merely by opening this notebook**.
- Keep the no-answer error analysis as a documented limitation; do not claim it has been resolved.